# NBA 3.1 – Rx patient journey (PJ03)

Same approach as the lab, Dx and Px notebooks: the 3.0 code below is kept unchanged as the baseline, then each step adds one change and is compared with the step before. The frozen 3.0 copy is `(Clone) 03_patient_journey_rx`; the plan and decisions D1–D8 are in `NBA_3_1_Rx_Journey_Plan.md` (approved 2026-10-29).

| Step | What | Status |
| --- | --- | --- |
| 3.0 | Original cells (Steps 1–6 of the 3.0 code), unchanged; only the cohort config path fixed (`../../`) | Baseline |
| 0 | 3.0 output vs the table last written (`kerendia_nba_3_1_rx_journey_validation`, 2026-10-26) | Built, identical |
| 1 | Hardcoded parts + profile: the product view, the typed DPP-IV / GLP-1 lists, the Rx basket, rows removed by each filter | Built |
| 2 | Basket: product list, journey category and brand label from the new Rx basket (`Kerendia_Rx_Final.csv`, 367 rows; original kept as `Kerendia NBA 3.1 - RX Event List.csv`) | Built, identical to 3.0 |
| 3 | Config: `RX_CONFIG`, one Snowflake query (160,029 rows pulled instead of 6.0 B), end = `date_end`, 97-day look-back | Built: identical with the 3.0 dates; end date −92 patient-months; look-back +2,525 patient-months in the first months |
| 4 | Code: plan and demographics joins removed, empty days supply dropped, dropped-rows report | Built: identical to Step 3; report adds up (158,983 fills) |
| 5 | Write: Step 4 output, same 5 columns; switched off (`WRITE_OUTPUT = False`); goes live with the 3.1 Dx / Px tables | Off |

**Output contract** (used by `06_patient_journey_combine`): `PATIENT_ID, DATE, TREATMENT_CATEGORY (array), CATEGORY (array), cohort_label`, same category texts (PJ06 looks for the exact text `SGLT2`).

## Patient Journey - Rx (3.0 header, kept as it was; the data sources and dates listed here are out of date, see the review in Step 1)
#### Data source used
* PHCDW.PHCDW_DSAA.IQV_LAAD_RX_VW
* PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT
* PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROVIDER

#### Steps:
1. Query rx claims data after 2017-01-01 for patient who visited neph after 2018-10-01
2. Get date level medication list (with 7-day additional window for gaps between refills) for each patient
3. Save table to databricks

#### Notes:
1. Treatment Cateogry and Brand

|TREATMENT_CATEGORY            |BRAND                         |
|------------------------------|------------------------------|
|ACE/ ARB                      |ACE/ ARB                      |
|Calcium channel / Beta blocker|Calcium channel / Beta blocker|
|Combo drug                    |Combo drug                    |
|Diuretic_NON_MRA              |Diuretic_NON_MRA              |
|Insulin                       |Insulin                       |
|metformin                     |metformin                     |
|MRA                           |MRA                           |
|SGLT2                         |INVOKANA 300MG                |
|SGLT2                         |FARXIGA                       |
|SGLT2                         |SGLT2                         |
|SGLT2                         |INVOKANA 100MG                |
|SGLT2                         |JARDIANCE                     |

* when PROD.PRODUCT_NAME like '%INVOKANA%' and  PRODUCT_STRENGTH = '100 MG' then 'INVOKANA 100MG'
* when PROD.PRODUCT_NAME like '%INVOKANA%' and  PRODUCT_STRENGTH = '300 MG' then 'INVOKANA 300MG'
* when PROD.PRODUCT_NAME like '%JARDIANCE%' then 'JARDIANCE'
* when PROD.PRODUCT_NAME like '%FARXIGA%' then 'FARXIGA'

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../../00_cohort_config"

## 2. Data cleanup and processing

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [0]:
# ===============================
# Step 1: Load / Assume tables as DataFrames
# ===============================
fact_rx_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX")
demographic_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC")
provider_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR")
product_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD")
plan_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PLN")

nba_hcp_df = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference").select('PROVIDER_ID').distinct()

ckd_mapping_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW")
print("Shape :", fact_rx_df.count(), len(fact_rx_df.columns))
print("Shape :", demographic_df.count(), len(demographic_df.columns))
print("Shape :", provider_df.count(), len(provider_df.columns))
print("Shape :", product_df.count(), len(product_df.columns))
print("Shape :", plan_df.count(), len(plan_df.columns))
print("Shape :", nba_hcp_df.count(), len(nba_hcp_df.columns))
print("Shape :", ckd_mapping_df.count(), len(ckd_mapping_df.columns))

In [0]:
# ===============================
# Step 2: Filter Fact RX & Join Dimensions
# ===============================
rx_base = (
    fact_rx_df.alias("f")
    .join(nba_hcp_df.alias("hcp"), F.col("f.PROVIDER_ID") == F.col("hcp.PROVIDER_ID"))
    .join(provider_df.alias("p"), F.col("f.PROVIDER_ID") == F.col("p.PROVIDER_ID"))
    .join(product_df.alias("prd"), F.col("f.NDC_CD") == F.col("prd.NDC_CD"))
    .join(plan_df.alias("pl"), F.col("f.PAYER_PLAN_ID") == F.col("pl.PAYER_PLAN_ID"))
    .join(demographic_df.alias("d"), F.col("f.PATIENT_ID") == F.col("d.PATIENT_ID"), "left")
    .join(ckd_mapping_df.alias("map"), F.col("prd.PRODUCT_NAME") == F.col("map.PRODUCT_NAME"))
    .filter(
        (F.col("f.SVC_DT") >= F.lit(DATA_START_DATE)) &
        (F.col("p.NPI_NUMBER").isNotNull()) &
        (~F.lower(F.col("map.Treatment_Category")).like("%other%"))
    )
    .select(
        "f.PATIENT_ID",
        "f.SVC_DT",
        "f.DAYS_SUPPLY",
        "prd.PRODUCT_NAME",
        "prd.PRODUCT_STRENGTH",
        "map.Treatment_Category",
        "p.NPI_NUMBER",
        "p.PRIMARY_SPECIALTY_DESC"
    )
)
# Validation filter: 10 test HCPs (from config)
rx_base, test_patients = apply_hcp_filter(rx_base)

print("Shape :", rx_base.count(), len(rx_base.columns))
rx_base.limit(100).display()

In [0]:
# ===============================
# Step 3: Compute END_DT per row
# ===============================
max_date = fact_rx_df.agg(F.max("SVC_DT").alias("max_date")).collect()[0]["max_date"]

rx_enriched = (
    rx_base
    .withColumn("DAYS_SUPPLY_T", F.round(F.col("DAYS_SUPPLY")).cast("int"))
    .withColumn("END_DT", F.least(
        F.expr("date_add(SVC_DT, DAYS_SUPPLY_T + 7)"),
        F.lit(max_date)
    ))
)
print("Shape :", rx_enriched.count(), len(rx_enriched.columns))
print("Max Date:", max_date)
rx_enriched.filter(F.col("PATIENT_ID") == "590252630").display()

In [0]:
# ===============================
# Step 4: Map Category using Spark SQL expressions (vectorized)
# ===============================
rx_enriched = rx_enriched.withColumn(
    "CATEGORY",
    F.when((F.col("PRODUCT_NAME").like("%INVOKANA%")) & (F.col("PRODUCT_STRENGTH") == "100 MG"), "INVOKANA 100MG")
     .when((F.col("PRODUCT_NAME").like("%INVOKANA%")) & (F.col("PRODUCT_STRENGTH") == "300 MG"), "INVOKANA 300MG")
     .when(F.col("PRODUCT_NAME").like("%JARDIANCE%"), "JARDIANCE")
     .when(F.col("PRODUCT_NAME").like("%FARXIGA%"), "FARXIGA")
     .when(F.col("PRODUCT_NAME").isin(
         'ALOGLIPTIN BENZOATE','ALOGLIPTIN-METFORMIN HCL','ALOGLIPTIN-PIOGLITAZONE',
         'SITAGLIPTIN-METFORMIN HCL','SITAGLIPTIN PHOSPHATE','LINAGLIPTIN-METFORMIN HCL',
         'SAXAGLIPTIN-METFORMIN HCL','SAXAGLIPTIN HCL','LINAGLIPTIN','ALOGLIPTIN',
         'ALOGLIPTIN/METFORMIN HCL','ALOGLIPTIN/PIOGLITAZONE','JANUMET','JANUMET XR',
         'JANUVIA','JENTADUETO','JENTADUETO XR','KAZANO','KOMBIGLYZE XR','NESINA',
         'ONGLYZA','OSENI','TRADJENTA'
     ), "DPP-IV")
     .when(F.col("PRODUCT_NAME").isin(
         'LIXISENATIDE','EXENATIDE','SEMAGLUTIDE','ALBIGLUTIDE','DULAGLUTIDE',
         'LIRAGLUTIDE','INSULIN DEGLUDEC-LIRAGLUTIDE','ADLYXIN','ADLYXIN STARTER PACK',
         'BYDUREON','BYDUREON BCISE','BYDUREON PEN','BYETTA','OZEMPIC','RYBELSUS',
         'TANZEUM','TRULICITY','VICTOZA','XULTOPHY 100/3.6'
     ), "GLP-1")
     .otherwise(F.col("Treatment_Category"))
)
print("Shape :", rx_enriched.count(), len(rx_enriched.columns))
rx_enriched.filter(F.col("PATIENT_ID") == "590252630").display()

In [0]:
# ===============================
# Step 5: Expand dates per day using explode(sequence())
# ===============================
rx_expanded = rx_enriched.withColumn(
    "TXN_DATE",
    F.explode(F.expr("sequence(SVC_DT, END_DT, interval 1 day)"))
)

# Convert to cohort-month granularity (using offset from config)
rx_expanded = rx_expanded.withColumn(
    "COHORT_MONTH",
    F.date_format(assign_cohort_month(F.col("TXN_DATE")), "yyyy-MM-dd")
)
print("Shape :", rx_expanded.count(), len(rx_expanded.columns))
rx_expanded.filter(F.col("PATIENT_ID") == "590252630").orderBy("COHORT_MONTH").display()

In [0]:
# ===============================
# Step 6: Aggregate per patient per month
# ===============================
final_df = (
    rx_expanded
    .groupBy("PATIENT_ID", "COHORT_MONTH")
    .agg(
        F.collect_set("Treatment_Category").alias("TREATMENT_CATEGORY"),
        F.collect_set("CATEGORY").alias("CATEGORY")
    )
    .orderBy("PATIENT_ID", "COHORT_MONTH")
)

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
final_df = final_df.join(
    df_cohort_ref.select(
        F.date_format("cohort_ref_month", "yyyy-MM-dd").alias("_ref_month_str"),
        "cohort_label"
    ),
    final_df.COHORT_MONTH == F.col("_ref_month_str"),
    "left"
).drop("_ref_month_str")

print("Shape :", final_df.count(), len(final_df.columns))
final_df.filter(F.col("PATIENT_ID") == "590252630").orderBy("COHORT_MONTH").display()


In [0]:
df_rx_clean = final_df

In [0]:
df_rx_clean = df_rx_clean.withColumnRenamed("COHORT_MONTH", "DATE")

In [0]:
# DATE is already in yyyy-MM-dd format from assign_cohort_month
df_rx_clean = df_rx_clean.withColumn("DATE", F.to_date(F.col("DATE"), "yyyy-MM-dd"))

In [0]:
print("Shape :", df_rx_clean.count(), len(df_rx_clean.columns))

## Step 0: Check the 3.0 baseline

The 3.0 cells above were run unchanged (only the cohort config path fixed). Their output (`df_rx_clean`, kept as `df_rx_30`) is compared with the table the 3.0 notebook last wrote (`kerendia_nba_3_1_rx_journey_validation`, 2026-10-26, 107,616 patient-months). Expected: identical, or differences explained by a data refresh.

The comparison function `compare_rx()` is reused in Steps 2–4: per patient-month, same / treatment categories differ / brand labels or cohort label differ / only new / only old; per category, patient-months old vs new; and patients with an SGLT2 month (the PJ06 SGLT2 flag).

In [0]:
# 0a. 3.0 output (run now) vs the table last written by the 3.0 notebook (2026-10-26). Validation HCPs only, as in 3.0.
import pandas as pd
RX_VALIDATION_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                      "hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_rx_journey_validation")
df_rx_30 = df_rx_clean   # the 3.0 output under its own name; the later steps are compared with it

def _norm_rx(df, sfx):
    # sorted arrays, so that the order inside collect_set does not count as a difference
    return df.select("PATIENT_ID", "DATE",
                     F.array_sort("TREATMENT_CATEGORY").alias(f"TC_{sfx}"),
                     F.array_sort("CATEGORY").alias(f"CAT_{sfx}"),
                     F.col("cohort_label").alias(f"LBL_{sfx}"),
                     F.lit(1).alias(f"IN_{sfx}"))

def compare_rx(df_old, df_new):
    # Returns (summary by result, patient-months per category old vs new, SGLT2 patients old vs new)
    j = _norm_rx(df_old, "O").join(_norm_rx(df_new, "N"), ["PATIENT_ID", "DATE"], "full_outer")
    res = (F.when(F.col("IN_O").isNull(), "4. only new")
            .when(F.col("IN_N").isNull(), "5. only old")
            .when((F.col("TC_O") == F.col("TC_N")) & (F.col("CAT_O") == F.col("CAT_N"))
                  & F.col("LBL_O").eqNullSafe(F.col("LBL_N")), "1. same")
            .when(F.col("TC_O") != F.col("TC_N"), "2. treatment categories differ")
            .otherwise("3. brand labels or cohort label differ"))
    summary = (j.withColumn("RESULT", res).groupBy("RESULT")
               .agg(F.count("*").alias("PATIENT_MONTHS"), F.countDistinct("PATIENT_ID").alias("PATIENTS"))
               .orderBy("RESULT").toPandas())

    def _per_cat(df, col, kind, who):
        return (df.select(F.explode(col).alias("VALUE")).groupBy("VALUE").count()
                .withColumnRenamed("count", who).withColumn("KIND", F.lit(kind)))
    cats = None
    for col, kind in [("TREATMENT_CATEGORY", "TREATMENT_CATEGORY"), ("CATEGORY", "CATEGORY (brand)")]:
        c = _per_cat(df_old, col, kind, "MONTHS_OLD").join(_per_cat(df_new, col, kind, "MONTHS_NEW"), ["KIND", "VALUE"], "full_outer")
        cats = c if cats is None else cats.unionByName(c)
    cats = cats.fillna(0).withColumn("CHANGE", F.col("MONTHS_NEW") - F.col("MONTHS_OLD")).orderBy("KIND", "VALUE").toPandas()

    _sg = lambda df: df.filter(F.array_contains("TREATMENT_CATEGORY", "SGLT2")).select("PATIENT_ID").distinct()
    so, sn = _sg(df_old), _sg(df_new)
    sglt2 = pd.DataFrame([{"SGLT2_PATIENTS_OLD": so.count(), "SGLT2_PATIENTS_NEW": sn.count(),
                           "BOTH": so.join(sn, "PATIENT_ID").count()}])
    return summary, cats, sglt2

df_rx_last = spark.read.format("delta").load(RX_VALIDATION_PATH)
print(f"Table last written: {df_rx_last.count():,} patient-months; 3.0 run now: {df_rx_30.count():,}")
_s0, _c0, _g0 = compare_rx(df_rx_last, df_rx_30)
print("Per patient-month (old = table last written, new = 3.0 run now)")
display(_s0)
print("Patient-months per category")
display(_c0)
print("Patients with an SGLT2 month (PJ06 SGLT2 flag)")
display(_g0)

### 0b. Findings: 3.0 baseline

Run 2026-10-29. **The 3.0 code gives exactly the same output as the table it wrote on 2026-10-26.**

| Check | Table last written | 3.0 run now |
| --- | --- | --- |
| Patient-months | 107,616 | 107,616 (all "same": same categories, brand labels and cohort label) |
| Patients | 3,645 | 3,645 |
| Fills used (after the joins and the validation filter) | – | 126,903 |
| Patients with an SGLT2 month (PJ06 SGLT2 flag) | 902 | 902 (the same 902) |

Patient-months per category are identical too. A few things this already shows:

* **The brand label `GLP-1` appears in only 14 patient-months, and `DPP-IV` never appears.** The typed lists only label products the view keeps, and the view puts most of them in an "other" category (see 1d).
* **`Insulin` as treatment category: 12,791 months; as brand label: 12,778.** The gap matches the 14 `GLP-1` months: Xultophy 100/3.6 (insulin + liraglutide) is the only typed GLP-1 name the view keeps (1d), so it is counted under an `Insulin` treatment category but labelled `GLP-1` as brand.
* **The `metformin` category has a trailing space (`"metformin "`)**, as it comes from the view. It must be kept exactly like this to keep parity with the table.
* The last Rx claim date (used as the end date) is 2026-10-02, the same as on 2026-10-26, so the Rx data has not been refreshed since.

## Step 1: Hardcoded parts and profile

**What the 3.0 code does:** takes Rx claims from HCPs in the universe since `DATA_START_DATE`, keeps the products found in the Snowflake view `CKD_PRODUCT_MAPPING_VW` (matched on product name), labels each fill with a category, spreads each fill over its days supply + 7 days, and outputs, per patient and cohort month, the set of categories the patient was on.

| # | Hardcoded today | Where it should live | Checked in |
| --- | --- | --- | --- |
| R1 | Which drugs count and their category: Snowflake view, matched on product name | Basket | 1b |
| R2 | Categories containing "other" excluded (text match) | Basket (active flag) | 1b, 1c |
| R3 | Brand labels: Invokana by strength text (`'100 MG'` / `'300 MG'`), Jardiance, Farxiga; 23 DPP-IV and 19 GLP-1 names typed in Step 4 | Basket (brand label, strength) | 1b, 1c |
| R4 | Newer GLP-1s (Mounjaro, Zepbound, Wegovy) not in the GLP-1 list | Phase B decision (D3) | 1b |
| R5 | The Rx basket (`Kerendia_Rx_Final.csv`) is not used; its groups do not match the journey categories | Basket (new columns) | 1b |
| R6 | Basket product names cut at 25 characters | Basket check | 1b, 1c |
| R7 | 7 grace days; end date = last date in the Rx table, not `date_end`; fills before `DATA_START_DATE` that cover it are not counted | Config | Step 3 |
| R8 | Unused patient demographics join; plan join (inner) may drop fills | Code | 1c |
| R9 | Five full tables pulled into Spark (`SELECT *`) | Config + code (one Snowflake query) | Step 3 |
| R10 | Table names, HCP universe path and output path typed in full | Config | Step 3 |
| R11 | No report of dropped rows | Code | 1c, Step 4 |
| R12 | SGLT2 combination products, new SGLT2s and Kerendia have no brand label of their own | Phase B decision (D3) | 1b |

**The profile (1b, 1c) runs in Snowflake for all US fills since `DATA_START_DATE`** (all HCPs; the HCP universe is a Delta table and is applied in Spark, so it is not part of these counts). The exact effect on the validation HCPs is measured in Steps 2–4.

| Cell | What it shows |
| --- | --- |
| 1b | Every product name in the view, the typed lists or the basket: view category, basket group, typed label, NDCs in the product table, fills and patients |
| 1c | Rows removed by each 3.0 filter; duplicate keys in the joined tables; product name length; Invokana strengths; empty days supply |
| 1d | Findings |

In [0]:
# 1b. Every product name in the view, in the typed DPP-IV / GLP-1 lists or in the Rx basket (clubbed rows expanded):
# view category, basket row and group, typed label, NDCs in the product table, fills and patients (all US, since DATA_START_DATE).
# The table names here are the 3.0 ones; they move to RX_CONFIG in Step 3.
RX_FACT_30 = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX"
RX_PRD_30 = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD"
RX_PRVDR_30 = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR"
RX_PLN_30 = "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PLN"
RX_VIEW_30 = "PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW"
RX_BASKET_FOLDER = ("/Workspace/Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_develop_hcp_claims/"
                    "NBA 3.1 - Dev/03_Feature_Engineering/03_HCP_level_features/Final Product Basket/")
RX_EVENT_LIST_PATH = RX_BASKET_FOLDER + "Kerendia NBA 3.1 - RX Event List.csv"   # original Rx basket, content unchanged (renamed 2026-10-29, D1)

# The typed lists of 3.0 Step 4
DPP4_NAMES_30 = ['ALOGLIPTIN BENZOATE', 'ALOGLIPTIN-METFORMIN HCL', 'ALOGLIPTIN-PIOGLITAZONE',
                 'SITAGLIPTIN-METFORMIN HCL', 'SITAGLIPTIN PHOSPHATE', 'LINAGLIPTIN-METFORMIN HCL',
                 'SAXAGLIPTIN-METFORMIN HCL', 'SAXAGLIPTIN HCL', 'LINAGLIPTIN', 'ALOGLIPTIN',
                 'ALOGLIPTIN/METFORMIN HCL', 'ALOGLIPTIN/PIOGLITAZONE', 'JANUMET', 'JANUMET XR',
                 'JANUVIA', 'JENTADUETO', 'JENTADUETO XR', 'KAZANO', 'KOMBIGLYZE XR', 'NESINA',
                 'ONGLYZA', 'OSENI', 'TRADJENTA']
GLP1_NAMES_30 = ['LIXISENATIDE', 'EXENATIDE', 'SEMAGLUTIDE', 'ALBIGLUTIDE', 'DULAGLUTIDE',
                 'LIRAGLUTIDE', 'INSULIN DEGLUDEC-LIRAGLUTIDE', 'ADLYXIN', 'ADLYXIN STARTER PACK',
                 'BYDUREON', 'BYDUREON BCISE', 'BYDUREON PEN', 'BYETTA', 'OZEMPIC', 'RYBELSUS',
                 'TANZEUM', 'TRULICITY', 'VICTOZA', 'XULTOPHY 100/3.6']

# Exact column names of the view (it may use quoted mixed-case names such as "Treatment_Category")
_vc = {c.upper(): c for c in Get_Data_Snowflakes(f"SELECT * FROM {RX_VIEW_30} LIMIT 1").columns}
V_NAME, V_CAT = f'"{_vc["PRODUCT_NAME"]}"', f'"{_vc["TREATMENT_CATEGORY"]}"'
print(f"View columns: {list(_vc.values())}")

# Original Rx basket, clubbed rows expanded to their member names
_rx_raw = pd.read_csv(RX_EVENT_LIST_PATH, dtype=str).fillna("")
_rows = []
for _, r in _rx_raw.iterrows():
    _names = ([m.strip() for m in r.club_members.split(",") if m.strip()] if r.is_clubbed.upper() == "YES"
              else [r.PRODUCT_NAME.strip()])
    _rows += [(n, r.PRODUCT_NAME, r.PRODUCT_GROUP) for n in _names]
rx_basket_names = pd.DataFrame(_rows, columns=["PRODUCT_NAME", "BASKET_ROW", "BASKET_GROUP"]).drop_duplicates("PRODUCT_NAME")
print(f"Rx basket: {len(_rx_raw)} rows ({(_rx_raw.is_clubbed.str.upper() == 'YES').sum()} clubbed) = {len(rx_basket_names)} product names")

_q = lambda s: "'" + str(s).replace("'", "''") + "'"
_b_values = ",\n".join(f"({_q(a)}, {_q(b)}, {_q(c)})" for a, b, c in rx_basket_names.itertuples(index=False))
_l_values = ",\n".join([f"({_q(n)}, 'DPP-IV')" for n in DPP4_NAMES_30] + [f"({_q(n)}, 'GLP-1')" for n in GLP1_NAMES_30])

rx_profile = Get_Data_Snowflakes(f"""
WITH V AS (SELECT {V_NAME} AS PRODUCT_NAME,
                  LISTAGG(DISTINCT {V_CAT}, ' | ') WITHIN GROUP (ORDER BY {V_CAT}) AS VIEW_CATEGORY,
                  COUNT(*) AS VIEW_ROWS
           FROM {RX_VIEW_30} GROUP BY 1),
     B AS (SELECT column1 AS PRODUCT_NAME, column2 AS BASKET_ROW, column3 AS BASKET_GROUP FROM VALUES
{_b_values}),
     L AS (SELECT column1 AS PRODUCT_NAME, column2 AS TYPED_LABEL FROM VALUES
{_l_values}),
     U AS (SELECT PRODUCT_NAME FROM V UNION SELECT PRODUCT_NAME FROM B UNION SELECT PRODUCT_NAME FROM L),
     P AS (SELECT NDC_CD, PRODUCT_NAME FROM {RX_PRD_30} WHERE PRODUCT_NAME IN (SELECT PRODUCT_NAME FROM U)),
     PN AS (SELECT PRODUCT_NAME, COUNT(DISTINCT NDC_CD) AS NDCS FROM P GROUP BY 1),
     S AS (SELECT P.PRODUCT_NAME, COUNT(*) AS RX_ROWS, COUNT(DISTINCT F.PATIENT_ID) AS PATIENTS
           FROM {RX_FACT_30} F JOIN P ON F.NDC_CD = P.NDC_CD
           WHERE F.SVC_DT >= '{DATA_START_DATE}' GROUP BY 1)
SELECT U.PRODUCT_NAME, V.VIEW_CATEGORY, V.VIEW_ROWS, B.BASKET_ROW, B.BASKET_GROUP, L.TYPED_LABEL,
       COALESCE(PN.NDCS, 0) AS NDCS, COALESCE(S.RX_ROWS, 0) AS RX_ROWS, COALESCE(S.PATIENTS, 0) AS PATIENTS
FROM U LEFT JOIN V ON U.PRODUCT_NAME = V.PRODUCT_NAME
LEFT JOIN B ON U.PRODUCT_NAME = B.PRODUCT_NAME
LEFT JOIN L ON U.PRODUCT_NAME = L.PRODUCT_NAME
LEFT JOIN PN ON U.PRODUCT_NAME = PN.PRODUCT_NAME
LEFT JOIN S ON U.PRODUCT_NAME = S.PRODUCT_NAME
ORDER BY PATIENTS DESC""").toPandas().fillna({"VIEW_CATEGORY": "", "BASKET_ROW": "", "BASKET_GROUP": "", "TYPED_LABEL": ""})
for _c in ["VIEW_ROWS", "NDCS", "RX_ROWS", "PATIENTS"]:
    rx_profile[_c] = pd.to_numeric(rx_profile[_c]).fillna(0).astype("int64")

# Flags: in the view; kept by 3.0 (at least one view category without "other"); in the basket; typed label
rx_profile["IN_VIEW"] = rx_profile.VIEW_CATEGORY != ""
import builtins   # any() may be shadowed by a Spark star import in the helper notebooks
rx_profile["KEPT_30"] = rx_profile.VIEW_CATEGORY.apply(lambda s: builtins.any("other" not in c.lower() for c in s.split(" | ") if c))
rx_profile["IN_BASKET"] = rx_profile.BASKET_ROW != ""

def _show(pdf):
    # display() cannot show an empty pandas table; print a line instead
    display(pdf) if len(pdf) else print("   (none)")
_cols = ["PRODUCT_NAME", "VIEW_CATEGORY", "BASKET_ROW", "BASKET_GROUP", "TYPED_LABEL", "NDCS", "RX_ROWS", "PATIENTS"]
_v = rx_profile[rx_profile.IN_VIEW]

print(f"A. View: {len(_v)} product names; kept by 3.0 (category without 'other'): {_v.KEPT_30.sum()}; with fills: {(_v.RX_ROWS > 0).sum()}")
display(_v.assign(CAT=_v.VIEW_CATEGORY).groupby(["CAT", "KEPT_30"])
        .agg(PRODUCTS=("PRODUCT_NAME", "size"), WITH_FILLS=("RX_ROWS", lambda s: (s > 0).sum()), FILLS=("RX_ROWS", "sum"))
        .reset_index().sort_values("FILLS", ascending=False))
print("B. View names with more than one row or category (3.0 join would repeat their fills):")
_show(_v[_v.VIEW_ROWS > 1][_cols + ["VIEW_ROWS"]])

print("C. Typed DPP-IV / GLP-1 names (3.0 Step 4): a name only gets its label if the view keeps it")
_t = rx_profile[rx_profile.TYPED_LABEL != ""]
display(_t.groupby("TYPED_LABEL").agg(NAMES=("PRODUCT_NAME", "size"), IN_VIEW=("IN_VIEW", "sum"), KEPT_30=("KEPT_30", "sum"),
                                      IN_PRODUCT_TABLE=("NDCS", lambda s: (s > 0).sum()), WITH_FILLS=("RX_ROWS", lambda s: (s > 0).sum())).reset_index())
print("   Typed names never labelled (not kept by the view):")
_show(_t[~_t.KEPT_30][_cols])

print("D. Products kept by 3.0 but NOT in the basket -> to add to the new basket (decision D2):")
_show(rx_profile[rx_profile.KEPT_30 & ~rx_profile.IN_BASKET][_cols])
print("E. Basket products NOT kept by 3.0 (not in the view, or 'other'), top 40 by patients -> ACTIVE_JOURNEY = N in Phase A:")
_show(rx_profile[rx_profile.IN_BASKET & ~rx_profile.KEPT_30].head(40)[_cols])
print("F. Basket names with no NDC in the product table (spelling or the 25-character cut):")
_show(rx_profile[rx_profile.IN_BASKET & (rx_profile.NDCS == 0)][_cols])

print("G. Drugs to watch (decision D3, Phase B): newer GLP-1s, SGLT2s and combos, MRAs, HF drugs")
_watch = ("MOUNJARO|ZEPBOUND|WEGOVY|TIRZEPATIDE|KERENDIA|FINERENONE|SPIRONOLACTONE|EPLERENONE|ENTRESTO|SACUBITRIL|"
          "SYNJARDY|XIGDUO|GLYXAMBI|TRIJARDY|INVOKAMET|QTERN|STEGLUJAN|SEGLUROMET|BRENZAVVY|INPEFA|STEGLATRO|"
          "INVOKANA|JARDIANCE|FARXIGA|DAPAGLIFLOZIN|EMPAGLIFLOZIN|CANAGLIFLOZIN")
_show(rx_profile[rx_profile.PRODUCT_NAME.str.contains(_watch, regex=True)][_cols])

In [0]:
# 1c. All US fills since DATA_START_DATE whose product is in the view (the candidates of the 3.0 code), by the first
# 3.0 rule that removes them. The HCP universe and the validation filter are applied in Spark and are not part of this.
rx_funnel = Get_Data_Snowflakes(f"""
WITH V AS (SELECT {V_NAME} AS PRODUCT_NAME,
                  MAX(IFF(LOWER({V_CAT}) NOT LIKE '%other%', 1, 0)) AS HAS_KEPT_CATEGORY
           FROM {RX_VIEW_30} GROUP BY 1),
     R AS (SELECT F.PATIENT_ID, F.PROVIDER_ID, F.PAYER_PLAN_ID, F.DAYS_SUPPLY, V.HAS_KEPT_CATEGORY
           FROM {RX_FACT_30} F
           JOIN {RX_PRD_30} P ON F.NDC_CD = P.NDC_CD
           JOIN V ON P.PRODUCT_NAME = V.PRODUCT_NAME
           WHERE F.SVC_DT >= '{DATA_START_DATE}'),
     PR AS (SELECT PROVIDER_ID, MAX(IFF(NPI_NUMBER IS NOT NULL, 1, 0)) AS HAS_NPI FROM {RX_PRVDR_30} GROUP BY 1),
     PL AS (SELECT DISTINCT PAYER_PLAN_ID FROM {RX_PLN_30})
SELECT CASE WHEN R.HAS_KEPT_CATEGORY = 0 THEN '1. view category contains "other"'
            WHEN PR.PROVIDER_ID IS NULL THEN '2. provider not in the provider table'
            WHEN PR.HAS_NPI = 0 THEN '3. provider has no NPI'
            WHEN PL.PAYER_PLAN_ID IS NULL THEN '4. plan not in the plan table'
            WHEN R.DAYS_SUPPLY IS NULL THEN '5. kept, but days supply empty'
            ELSE '6. kept' END AS REASON,
       COUNT(*) AS ROWS_, COUNT(DISTINCT R.PATIENT_ID) AS PATIENTS
FROM R LEFT JOIN PR ON R.PROVIDER_ID = PR.PROVIDER_ID
LEFT JOIN PL ON R.PAYER_PLAN_ID = PL.PAYER_PLAN_ID
GROUP BY 1 ORDER BY 1""").toPandas()
rx_funnel["ROWS_"] = rx_funnel["ROWS_"].astype("int64")
rx_funnel["SHARE_ROWS_PCT"] = (100 * rx_funnel["ROWS_"] / rx_funnel["ROWS_"].sum()).round(3)
print(f"Fills with a view product since {DATA_START_DATE} (all US, all HCPs): {rx_funnel.ROWS_.sum():,}")
display(rx_funnel)

# Duplicate keys in the tables the 3.0 code joins (a duplicate key repeats the fill rows)
print("Join keys: rows vs distinct keys")
display(Get_Data_Snowflakes(f"""
SELECT 'product table: NDC_CD' AS TABLE_KEY, COUNT(*) AS ROWS_, COUNT(DISTINCT NDC_CD) AS KEYS FROM {RX_PRD_30}
UNION ALL SELECT 'provider table: PROVIDER_ID', COUNT(*), COUNT(DISTINCT PROVIDER_ID) FROM {RX_PRVDR_30}
UNION ALL SELECT 'plan table: PAYER_PLAN_ID', COUNT(*), COUNT(DISTINCT PAYER_PLAN_ID) FROM {RX_PLN_30}
UNION ALL SELECT 'view: PRODUCT_NAME', COUNT(*), COUNT(DISTINCT {V_NAME}) FROM {RX_VIEW_30}"""))

# Product name length in the product table (are names cut at 25 characters there too, as in the basket?)
print("Product names in the product table: length")
display(Get_Data_Snowflakes(f"""
SELECT COUNT(DISTINCT PRODUCT_NAME) AS NAMES, MAX(LENGTH(PRODUCT_NAME)) AS MAX_LENGTH,
       COUNT(DISTINCT IFF(LENGTH(PRODUCT_NAME) = 25, PRODUCT_NAME, NULL)) AS NAMES_25_CHARS,
       COUNT(DISTINCT IFF(LENGTH(PRODUCT_NAME) > 25, PRODUCT_NAME, NULL)) AS NAMES_OVER_25
FROM {RX_PRD_30}"""))

# Invokana strengths: 3.0 labels only the exact texts '100 MG' and '300 MG'
print("Invokana: strength texts in the product table and fills since the start date")
display(Get_Data_Snowflakes(f"""
SELECT P.PRODUCT_NAME, P.PRODUCT_STRENGTH, COUNT(F.NDC_CD) AS RX_ROWS
FROM {RX_PRD_30} P LEFT JOIN {RX_FACT_30} F ON F.NDC_CD = P.NDC_CD AND F.SVC_DT >= '{DATA_START_DATE}'
WHERE P.PRODUCT_NAME LIKE '%INVOKANA%' GROUP BY 1, 2 ORDER BY 1, 2"""))

# Days supply of the kept fills: empty, zero, and the largest values (the end date is SVC_DT + days supply + 7)
print("Days supply (fills with a view product, since the start date)")
display(Get_Data_Snowflakes(f"""
SELECT COUNT(*) AS ROWS_, COUNT_IF(F.DAYS_SUPPLY IS NULL) AS EMPTY, COUNT_IF(F.DAYS_SUPPLY = 0) AS ZERO,
       COUNT_IF(F.DAYS_SUPPLY > 90) AS OVER_90, COUNT_IF(F.DAYS_SUPPLY > 365) AS OVER_365, MAX(F.DAYS_SUPPLY) AS MAX_DAYS,
       MEDIAN(F.DAYS_SUPPLY) AS MEDIAN_DAYS
FROM {RX_FACT_30} F JOIN {RX_PRD_30} P ON F.NDC_CD = P.NDC_CD
JOIN (SELECT DISTINCT {V_NAME} AS PRODUCT_NAME FROM {RX_VIEW_30}) V ON P.PRODUCT_NAME = V.PRODUCT_NAME
WHERE F.SVC_DT >= '{DATA_START_DATE}'"""))

In [0]:
# 1c-2. In the 3.0 code an empty DAYS_SUPPLY gives date_add(SVC_DT, null) = null, and least(null, max_date) = max_date
# (Spark's least() skips nulls), so such a fill is spread to the end of the data. Check on the 3.0 run (validation HCPs).
_ds = (rx_enriched
       .withColumn("DAYS_SPREAD", F.datediff("END_DT", "SVC_DT") + 1)
       .withColumn("GROUP", F.when(F.col("DAYS_SUPPLY_T").isNull(), "1. days supply empty")
                            .when(F.col("DAYS_SUPPLY_T") == 0, "2. days supply 0")
                            .when(F.col("DAYS_SUPPLY_T") > 365, "4. days supply > 365")
                            .when(F.col("DAYS_SUPPLY_T") > 90, "3. days supply 91-365")
                            .otherwise("0. days supply 1-90")))
display(_ds.groupBy("GROUP").agg(F.count("*").alias("FILLS"), F.countDistinct("PATIENT_ID").alias("PATIENTS"),
                                 F.min("DAYS_SPREAD").alias("MIN_DAYS_SPREAD"), F.max("DAYS_SPREAD").alias("MAX_DAYS_SPREAD"),
                                 F.sum("DAYS_SPREAD").alias("PATIENT_DAYS")).orderBy("GROUP"))
print("Fills with empty days supply (all of them, validation HCPs):")
_e = _ds.filter(F.col("DAYS_SUPPLY_T").isNull()).select("PATIENT_ID", "SVC_DT", "DAYS_SUPPLY", "PRODUCT_NAME", "Treatment_Category", "END_DT", "DAYS_SPREAD")
_show(_e.orderBy("SVC_DT").toPandas())

### 1d. Findings: profile

Run 2026-10-29, all US fills since 2023-01-01 (Snowflake), plus the 3.0 run for the validation HCPs (1c-2).

**In short:** the 3.0 code does what it says, nothing is lost by accident in the joins, and the plan join can go. But the view decides a lot more than it seems: it keeps only 254 of its 476 product names, and several big CKD / diabetes drugs never reach the journey (Kerendia, Ozempic, Mounjaro, generic dapagliflozin, nebivolol). Phase A keeps this as it is (parity); these are the facts for the Phase B decisions.

#### 1. What the view keeps

The view `CKD_PRODUCT_MAPPING_VW` has 476 product names (one row each, no duplicates), with columns `PRODUCT_NAME, GENERIC_NAME, TREATMENT_CATEGORY`. The 3.0 code keeps a name when its category does not contain "other".

| View category | Product names | With fills | Fills since 2023 | Kept by 3.0 |
| --- | --- | --- | --- | --- |
| Calcium channel / Beta blocker | 56 | 53 | 1,031 M | yes |
| ACE/ ARB | 32 | 32 | 593 M | yes |
| Combo drug | 45 | 39 | 422 M | yes |
| Diuretic_NON_MRA | 30 | 27 | 386 M | yes |
| `metformin ` | 6 | 6 | 336 M | yes |
| Insulin | 67 | 46 | 204 M | yes |
| SGLT2 | 13 | 13 | 163 M | yes |
| MRA | 5 | 5 | 108 M | yes |
| **Other- glucose mgmt** | 46 | 46 | 391 M | **no** |
| **Other** | 176 | 77 | 107 M | **no** |

So 254 names are kept, 222 dropped. "Other- glucose mgmt" holds the GLP-1 and DPP-IV products that are in the view (Ozempic, Trulicity, Rybelsus, Victoza, Januvia, Tradjenta, Janumet) and the sulfonylureas and pioglitazone.

#### 2. The typed DPP-IV / GLP-1 lists (3.0 Step 4)

| List | Names | In the view | Kept by the view | In the product table |
| --- | --- | --- | --- | --- |
| DPP-IV | 23 | 14 | **0** | 15 |
| GLP-1 | 19 | 8 | **1** (Xultophy 100/3.6) | 14 |

* **No DPP-IV name and only one GLP-1 name can ever be labelled.** The list is reached only after the view filter, and the view puts these drugs in "Other- glucose mgmt". 41 of the 42 typed names are never used.
* 13 typed names (e.g. `SEMAGLUTIDE`, `DULAGLUTIDE`, `LINAGLIPTIN`, `SITAGLIPTIN-METFORMIN HCL`) do not exist in the product table at all: the product table has brand or other spellings.
* Phase A keeps this as is (the label lists move to the basket unchanged). Labelling GLP-1 / DPP-IV for real is a Phase B decision (D3).

#### 3. View products missing from the Rx basket (to add, D2)

**94 names the 3.0 code keeps are not in the Rx basket.** 61 have fills, 33 have none. These must be added to the new basket so that Step 2 gives the same output. The largest:

| Product | View category | Patients since 2023 |
| --- | --- | --- |
| INDAPAMIDE | Diuretic_NON_MRA | 289,510 |
| CARDIZEM LA, CARDIZEM CD, COREG, COREG CR, LOPRESSOR, INNOPRAN XL | Calcium channel / Beta blocker | 5,500–18,000 each |
| GLUMETZA, GLUCOPHAGE, GLUCOPHAGE XR | `metformin ` | 4,550–16,158 each |
| AMLODIPINE/VALSARTAN/HCTZ, AVAPRO, ATACAND, TRIBENZOR | ACE/ ARB | 3,600–7,500 each |
| QUINAPRIL/HYDROCHLOROTHIA, HYZAAR, EXFORGE, AZOR, EXFORGE HCT | Combo drug | 3,600–6,900 each |

Mostly brand names of old generics and combinations. The 33 names with no fills are Humulin / Novolin insulins, hydralazine, clonidine and prazosin (all "Combo drug" in the view), Omnipod packs, etc. They have no NDC in the product table under that name, so adding them changes nothing. They are added only to keep the full view in the basket.

#### 4. Basket products the 3.0 code does not keep (`ACTIVE_JOURNEY = N`)

The Rx basket has 258 rows (28 clubbed) = 346 product names. **160 are kept by the 3.0 code, 186 are not** (not in the view, or in an "other" category). The largest, by patients since 2023:

| Product | Basket group | View | Patients |
| --- | --- | --- | --- |
| OZEMPIC | OZEMPIC | Other- glucose mgmt | 13.2 M |
| MOUNJARO | MOUNJARO | not in the view | 11.0 M |
| DAPAGLIFLOZIN (generic Farxiga) | FARXIGA GENERIC | **not in the view** | 4.3 M |
| WEGOVY, ZEPBOUND | WEGOVY, ZEPBOUND | not in the view | 4.3 M, 4.1 M |
| TRULICITY | TRULICITY | Other- glucose mgmt | 4.2 M |
| GLIPIZIDE, GLIMEPIRIDE | OTHERS | Other- glucose mgmt | 3.6 M, 3.3 M |
| Dexcom / FreeStyle Libre sensors | CLUBBED (CGM) | not in the view or Other | up to 3.4 M |
| INSULIN GLARGINE-YFGN, INSULIN GLARGINE SOLOSTAR | CLUBBED (Lantus family) | **not in the view** | 2.5 M, 1.2 M |
| SACUBITRIL/VALSARTAN (generic Entresto) | ENTRESTO GENERIC | **not in the view** | 1.7 M |
| NEBIVOLOL HYDROCHLORIDE | BB | **not in the view** | 1.5 M |
| KERENDIA | KERENDIA | **not in the view** | 446 K |

The ones in bold look like gaps in the view rather than choices. Their brand versions or class neighbours are kept (Farxiga → SGLT2, Entresto → ACE/ ARB, other beta blockers, 46 insulin names), but these generics or the drug itself are not. **Kerendia never appears in the Rx journey**, and generic dapagliflozin patients do not get the PJ06 SGLT2 flag.

#### 5. The 25-character question

No issue. The IQVIA product table itself stores names of at most 25 characters (610 names, 111 of exactly 25, none longer). Every basket name, clubbed members included, matches a product table name (0 basket names without an NDC). Matching on the name works.

#### 6. Rows removed by each 3.0 filter (all US, since 2023)

Of the **3,740,781,815 fills** of a view product (the rows below add up to this total):

| First rule that removes the fill | Fills | Share | Patients |
| --- | --- | --- | --- |
| View category contains "other" | 498,571,311 | 13.3% | 32.6 M |
| Provider not in the provider table | 44 | 0.0% | 32 |
| Provider has no NPI | 12,094,572 | 0.3% | 3.2 M |
| **Plan not in the plan table (plan join, D6)** | **0** | 0% | 0 |
| Kept, but days supply empty | 2,686 | 0.0% | 2,273 |
| Kept | 3,230,113,202 | 86.3% | 130.9 M |

* **The plan join removes nothing.** Every fill has a plan in the plan table, and no column of the plan table is used, so it can be removed (D6). The demographics join (left join, nothing used) can go too.
* **No join repeats rows.** Product, provider, plan and view all have one row per key.
* The HCP universe (applied in Spark) is not part of these counts; its effect is part of the Step 2–4 comparisons.

#### 7. Days supply

| Days supply | All US fills of a view product | Validation HCPs (3.0 run, 126,903 fills) |
| --- | --- | --- |
| Empty | 3,750 (2,686 of them kept) | 0 |
| 0 | 1,200,149 | 28 fills, each spread over 8 days (0 + 7 grace + the day itself) |
| 1–90 | most (median 84) | 125,006 |
| 91–365 | 52,605,134 | 1,865 |
| Over 365 (max 999) | 57,982 | 4 fills, spread over 383–913 days |

* **Empty days supply:** in the 3.0 code `date_add(SVC_DT, null)` is null, and Spark's `least(null, max_date)` returns `max_date`. So **such a fill is spread from its date to the end of the data** (2026-10-02), possibly years. It does not occur for the validation HCPs today (0 fills), but it would in the all-US run (2,686 kept fills, 2,273 patients). Proposal for Step 4: drop these fills and show them in the dropped-rows report.
* Very long days supply (up to 999 days) is spread in full. Only 4 validation fills are affected. A cap is a Phase B question, not changed in Phase A.

#### 8. Invokana strengths

Only two strength texts exist for INVOKANA: `100 MG` (994,088 fills) and `300 MG` (1,275,307 fills). So the 3.0 rule labels every Invokana fill. Invokamet and Invokamet XR do not match `%INVOKANA%` and keep the label `SGLT2`. The strength rule goes into the basket `STRENGTH` column unchanged.

#### 9. Drugs to watch (D3, Phase B)

| Drug | In the view? | Patients since 2023 | In the 3.0 journey as |
| --- | --- | --- | --- |
| Mounjaro, Zepbound, Wegovy, Wegovy HD | no | 11.0 M, 4.1 M, 4.3 M, 40 K | not counted |
| Ozempic, Trulicity, Rybelsus, Victoza | Other- glucose mgmt | 13.2 M, 4.2 M, 2.1 M, 0.9 M | not counted |
| Kerendia | no | 446 K | not counted |
| Spironolactone, eplerenone | MRA | 12.8 M, 352 K | MRA |
| Spironolactone/HCTZ | Diuretic_NON_MRA | 193 K | Diuretic_NON_MRA |
| Jardiance, Farxiga, Invokana, Steglatro | SGLT2 | 10.3 M, 5.1 M, 282 K, 190 K | SGLT2 with brand label (Jardiance, Farxiga, Invokana 100/300 MG); Steglatro as SGLT2 |
| Dapagliflozin (generic), Brenzavvy, Inpefa | no | 4.3 M, 22 K, 13 K | not counted |
| Synjardy (XR), Xigduo XR, Glyxambi, Invokamet (XR), Segluromet, Steglujan, Qtern | SGLT2 | 13 K–444 K | SGLT2 (label `SGLT2`) |
| Trijardy XR, dapagliflozin/metformin | no | 98 K, 174 K | not counted |
| Entresto | ACE/ ARB | 2.2 M | ACE/ ARB |
| Sacubitril/valsartan (generic) | no | 1.7 M | not counted |

The biggest open points for Phase B (D3): **Kerendia itself, generic dapagliflozin (changes the PJ06 SGLT2 flag), the GLP-1 / GIP-GLP-1 drugs, and generic Entresto.**

## Step 2: Drug rules from the Rx basket

**One change only:** which products count, their journey category and their brand label come from the new Rx basket instead of the view `CKD_PRODUCT_MAPPING_VW` + the typed lists in 3.0 Step 4. Everything else (tables, joins, dates, grace days, spreading, cohort months) is the 3.0 code unchanged. Expected: identical output.

| Cell | What |
| --- | --- |
| 2a | Builds the new basket once from the original basket + the view + the 3.0 rules (one-off, switched off after the run). The original basket (Sehar's) keeps its content and is renamed `Kerendia NBA 3.1 - RX Event List.csv`; the new basket is `Kerendia_Rx_Final.csv` (D1, same naming as Lab, Dx and Px) |
| 2b | Reads and checks the new basket (this is what the notebook uses from now on) |
| 2c | Runs the journey with the basket (validation HCPs) |
| 2d | Compares with 3.0: patient-months (validation HCPs) and, for every product in the IQVIA product table, kept / category / label (covers all US fills) |
| 2e | Findings |

**New basket format** (plan section 5): `MATCH_TYPE` (`NAME` or `LIST`), `MATCH_VALUE` (names separated by `|`), `CLUB_ID`, `PRODUCT_GROUP`, `TREATMENT_CATEGORY` (3.0 text, empty when not in the journey), `BRAND_LABEL` (empty = use the category), `STRENGTH` (only for label rows that depend on it: Invokana 100 / 300 MG), `CONDITION` (empty in Phase A; filled in Phase B), `ACTIVE_JOURNEY`, `PATIENTS_TOTAL`, `IN_752_FEATURES`, `NOTE`.

A clubbed row whose members have different 3.0 categories is split into one row per category (same `CLUB_ID`, `NOTE` says why), so that every row has one rule.

In [0]:
# 2a. One-off: build the new Rx basket from the original basket + the view + the 3.0 Step 4 rules, so that it
# reproduces 3.0 exactly (decision D2). Needs rx_profile / rx_basket_names / _rx_raw from 1b.
# Switched off after the run: from now on the basket file is the source and is edited by hand.
import csv
BUILD_RX_BASKET = False   # built once on 2026-10-29 (367 rows); keep False: the basket file is now the source
RX_BASKET_PATH = RX_BASKET_FOLDER + "Kerendia_Rx_Final.csv"   # new-format Rx basket (built here as ..._3_1_draft.csv, renamed 2026-10-29, D1)

def _label_30(name, strength=None):
    # 3.0 Step 4, same order; "" = no label of its own (the category is used)
    if "INVOKANA" in name and strength == "100 MG": return "INVOKANA 100MG"
    if "INVOKANA" in name and strength == "300 MG": return "INVOKANA 300MG"
    if "JARDIANCE" in name: return "JARDIANCE"
    if "FARXIGA" in name: return "FARXIGA"
    if name in DPP4_NAMES_30: return "DPP-IV"
    if name in GLP1_NAMES_30: return "GLP-1"
    return ""

if BUILD_RX_BASKET:
    _p = rx_profile.set_index("PRODUCT_NAME")
    def _rule(name):
        # (ACTIVE_JOURNEY, TREATMENT_CATEGORY, BRAND_LABEL, NOTE) of one product name under the 3.0 code
        if name in _p.index and _p.at[name, "KEPT_30"]:
            return ("Y", _p.at[name, "VIEW_CATEGORY"], _label_30(name), "")
        if name in _p.index and _p.at[name, "IN_VIEW"]:
            return ("N", "", "", f"Not in the 3.0 journey: view category '{_p.at[name, 'VIEW_CATEGORY']}'")
        return ("N", "", "", "Not in the 3.0 journey: not in the view")

    _out = []
    def _add(match_type, names, club, group, rule, patients, in752, note):
        act, cat, lab, why = rule
        _out.append(dict(MATCH_TYPE=match_type, MATCH_VALUE="|".join(names), CLUB_ID=club, PRODUCT_GROUP=group,
                         TREATMENT_CATEGORY=cat, BRAND_LABEL=lab, STRENGTH="", CONDITION="", ACTIVE_JOURNEY=act,
                         PATIENTS_TOTAL=patients, IN_752_FEATURES=in752, NOTE="; ".join(x for x in [note, why] if x)))
        # Strength-dependent label rows (3.0: Invokana 100 / 300 MG); they only change the label of an active name
        if act == "Y":
            for n in names:
                for s in ["100 MG", "300 MG"]:
                    if _label_30(n, s) != _label_30(n):
                        _out.append(dict(MATCH_TYPE="NAME", MATCH_VALUE=n, CLUB_ID=club, PRODUCT_GROUP=group,
                                         TREATMENT_CATEGORY=cat, BRAND_LABEL=_label_30(n, s), STRENGTH=s, CONDITION="",
                                         ACTIVE_JOURNEY="Y", PATIENTS_TOTAL="", IN_752_FEATURES="",
                                         NOTE=f"Brand label by strength (3.0 rule): {s}"))

    # 1. All rows of the original basket (clubbed rows split when their members have different 3.0 rules)
    for _, r in _rx_raw.iterrows():
        clubbed = r.is_clubbed.upper() == "YES"
        names = [m.strip() for m in r.club_members.split(",") if m.strip()] if clubbed else [r.PRODUCT_NAME.strip()]
        groups = {}
        for n in names:
            groups.setdefault(_rule(n), []).append(n)
        split = len(groups) > 1
        for rule, members in groups.items():
            _add("LIST" if clubbed else "NAME", members, r.PRODUCT_NAME if clubbed else "", r.PRODUCT_GROUP, rule,
                 r.patients_total, r.in_752_features,
                 "Split (D2): club members have different 3.0 categories" if split else "")

    # 2. Products the 3.0 code keeps that are not in the original basket (D2)
    for n in rx_profile[rx_profile.KEPT_30 & ~rx_profile.IN_BASKET].PRODUCT_NAME:
        _add("NAME", [n], "", "", _rule(n), "", "", "Added (D2): in the view, not in the basket")

    rx_basket_new = pd.DataFrame(_out)
    # QUOTE_ALL keeps the trailing space of the 3.0 category "metformin " safe in editors
    rx_basket_new.to_csv(RX_BASKET_PATH, index=False, quoting=csv.QUOTE_ALL)
    print(f"Written: {RX_BASKET_PATH}")
    print(f"Rows: {len(rx_basket_new)} (original rows: {len(_rx_raw)}; added (D2): {rx_basket_new.NOTE.str.startswith('Added').sum()}; "
          f"strength label rows: {(rx_basket_new.STRENGTH != '').sum()}; rows from split clubs: {rx_basket_new.NOTE.str.contains('Split').sum()})")
    display(rx_basket_new.groupby(["ACTIVE_JOURNEY", "TREATMENT_CATEGORY"]).size().reset_index(name="ROWS"))
else:
    print(f"BUILD_RX_BASKET = False: basket not rebuilt ({RX_BASKET_PATH}).")

In [0]:
# 2b. Read the new Rx basket and check it. Returns:
#   base:      one row per active product name -> TREATMENT_CATEGORY, BRAND_LABEL (rows without STRENGTH)
#   overrides: brand label for (product name, strength), e.g. INVOKANA 100 MG
#   names:     every product name in the basket (all rows expanded), for reports
RX_BASKET_COLUMNS = ["MATCH_TYPE", "MATCH_VALUE", "CLUB_ID", "PRODUCT_GROUP", "TREATMENT_CATEGORY", "BRAND_LABEL",
                     "STRENGTH", "CONDITION", "ACTIVE_JOURNEY", "PATIENTS_TOTAL", "IN_752_FEATURES", "NOTE"]

def read_rx_basket(path):
    b = pd.read_csv(path, dtype=str, keep_default_na=False)   # keep_default_na: empty stays "", spaces are kept
    missing = [c for c in RX_BASKET_COLUMNS if c not in b.columns]
    assert not missing, f"Basket columns missing: {missing}"
    assert b.MATCH_TYPE.isin(["NAME", "LIST"]).all(), "MATCH_TYPE must be NAME or LIST"
    assert b.ACTIVE_JOURNEY.isin(["Y", "N"]).all(), "ACTIVE_JOURNEY must be Y or N"
    bad = b[(b.ACTIVE_JOURNEY == "Y") & (b.TREATMENT_CATEGORY == "")]
    assert bad.empty, f"Active rows without TREATMENT_CATEGORY:\n{bad}"
    rows = []
    for _, r in b.iterrows():
        for n in (r.MATCH_VALUE.split("|") if r.MATCH_TYPE == "LIST" else [r.MATCH_VALUE]):
            rows.append(dict(PRODUCT_NAME=n.strip(), STRENGTH=r.STRENGTH, TREATMENT_CATEGORY=r.TREATMENT_CATEGORY,
                             BRAND_LABEL=r.BRAND_LABEL, ACTIVE_JOURNEY=r.ACTIVE_JOURNEY, CLUB_ID=r.CLUB_ID,
                             PRODUCT_GROUP=r.PRODUCT_GROUP))
    names = pd.DataFrame(rows)
    base_all = names[names.STRENGTH == ""]
    dup = base_all[base_all.duplicated("PRODUCT_NAME", keep=False)]
    assert dup.empty, f"Product names on more than one row:\n{dup}"
    base = base_all[base_all.ACTIVE_JOURNEY == "Y"][["PRODUCT_NAME", "TREATMENT_CATEGORY", "BRAND_LABEL"]].reset_index(drop=True)
    overrides = names[(names.STRENGTH != "") & (names.ACTIVE_JOURNEY == "Y")][["PRODUCT_NAME", "STRENGTH", "BRAND_LABEL"]].reset_index(drop=True)
    assert not overrides.duplicated(["PRODUCT_NAME", "STRENGTH"]).any(), "A (name, strength) label is on more than one row"
    orphan = set(overrides.PRODUCT_NAME) - set(base.PRODUCT_NAME)
    assert not orphan, f"Strength label rows without an active product row: {orphan}"
    return base, overrides, names

rx_basket_base, rx_basket_overrides, rx_basket_all = read_rx_basket(RX_BASKET_PATH)
print(f"Basket: {RX_BASKET_PATH}")
print(f"Product names: {rx_basket_all.PRODUCT_NAME.nunique()}; active in the journey: {len(rx_basket_base)}; strength label rows: {len(rx_basket_overrides)}")
display(rx_basket_base.groupby(["TREATMENT_CATEGORY", "BRAND_LABEL"]).size().reset_index(name="PRODUCT_NAMES"))
display(rx_basket_overrides)

In [0]:
# 2c. Same as 3.0 Steps 2-6, with the view join and the Step 4 labels replaced by the basket.
from pyspark.sql.types import StructType, StructField, StringType

def rx_month_journey(rx_fills, end_date, grace_days=7):
    # 3.0 Steps 3, 5, 6 unchanged: END_DT = SVC_DT + days supply + grace days (capped at end_date), one row per day,
    # cohort month, set of categories and labels per patient and month, cohort label, DATE as date.
    e = (rx_fills.withColumn("DAYS_SUPPLY_T", F.round(F.col("DAYS_SUPPLY")).cast("int"))
                 .withColumn("END_DT", F.least(F.expr(f"date_add(SVC_DT, DAYS_SUPPLY_T + {grace_days})"), F.lit(end_date))))
    x = (e.withColumn("TXN_DATE", F.explode(F.expr("sequence(SVC_DT, END_DT, interval 1 day)")))
          .withColumn("COHORT_MONTH", F.date_format(assign_cohort_month(F.col("TXN_DATE")), "yyyy-MM-dd")))
    m = (x.groupBy("PATIENT_ID", "COHORT_MONTH")
          .agg(F.collect_set("Treatment_Category").alias("TREATMENT_CATEGORY"), F.collect_set("CATEGORY").alias("CATEGORY")))
    m = m.join(df_cohort_ref.select(F.date_format("cohort_ref_month", "yyyy-MM-dd").alias("_ref_month_str"), "cohort_label"),
               m.COHORT_MONTH == F.col("_ref_month_str"), "left").drop("_ref_month_str")
    return m.withColumnRenamed("COHORT_MONTH", "DATE").withColumn("DATE", F.to_date("DATE", "yyyy-MM-dd"))

_b = spark.createDataFrame(rx_basket_base, schema=StructType([StructField(c, StringType()) for c in ["PRODUCT_NAME", "TREATMENT_CATEGORY", "BRAND_LABEL"]]))
_o = spark.createDataFrame(rx_basket_overrides, schema=StructType([StructField(c, StringType()) for c in ["PRODUCT_NAME", "STRENGTH", "BRAND_LABEL"]]))

rx_fills_s2 = (
    fact_rx_df.alias("f")
    .join(nba_hcp_df.alias("hcp"), F.col("f.PROVIDER_ID") == F.col("hcp.PROVIDER_ID"))
    .join(provider_df.alias("p"), F.col("f.PROVIDER_ID") == F.col("p.PROVIDER_ID"))
    .join(product_df.alias("prd"), F.col("f.NDC_CD") == F.col("prd.NDC_CD"))
    .join(plan_df.alias("pl"), F.col("f.PAYER_PLAN_ID") == F.col("pl.PAYER_PLAN_ID"))          # unchanged until Step 4 (D6)
    .join(demographic_df.alias("d"), F.col("f.PATIENT_ID") == F.col("d.PATIENT_ID"), "left")    # unchanged until Step 4 (D6)
    .join(_b.alias("b"), F.col("prd.PRODUCT_NAME") == F.col("b.PRODUCT_NAME"))                 # was: the view + "other" filter
    .join(_o.alias("o"), (F.col("prd.PRODUCT_NAME") == F.col("o.PRODUCT_NAME"))
                         & (F.col("prd.PRODUCT_STRENGTH") == F.col("o.STRENGTH")), "left")
    .filter((F.col("f.SVC_DT") >= F.lit(DATA_START_DATE)) & F.col("p.NPI_NUMBER").isNotNull())
    .select("f.PATIENT_ID", "f.SVC_DT", "f.DAYS_SUPPLY", "prd.PRODUCT_NAME", "prd.PRODUCT_STRENGTH",
            F.col("b.TREATMENT_CATEGORY").alias("Treatment_Category"),
            # was: 3.0 Step 4. Label = strength row, else the basket label, else the category
            F.coalesce(F.expr("nullif(o.BRAND_LABEL, '')"), F.expr("nullif(b.BRAND_LABEL, '')"), F.col("b.TREATMENT_CATEGORY")).alias("CATEGORY"),
            "p.NPI_NUMBER", "p.PRIMARY_SPECIALTY_DESC")
)
rx_fills_s2, _ = apply_hcp_filter(rx_fills_s2)
df_rx_s2 = rx_month_journey(rx_fills_s2, max_date)   # same end date as 3.0 (last Rx claim date) until Step 3
print(f"Fills: Step 2 {rx_fills_s2.count():,} vs 3.0 {rx_enriched.count():,}")
print(f"Patient-months: Step 2 {df_rx_s2.count():,} vs 3.0 {df_rx_30.count():,}")

In [0]:
# 2d. (1) Validation HCPs: the patient-month output, as PJ06 uses it.
_s2, _c2, _g2 = compare_rx(df_rx_30, df_rx_s2)
print("Validation HCPs, per patient-month (old = 3.0, new = Step 2)")
display(_s2)
display(_c2[_c2.CHANGE != 0] if (_c2.CHANGE != 0).any() else pd.DataFrame([{"PER_CATEGORY": "no change in any category"}]))
display(_g2)

# (2) All US: every (product name, strength) in the IQVIA product table, 3.0 rule vs basket rule.
# Every fill reaches the journey through one of these, so equal rules here = equal fills for all US.
_lab30 = (F.when(F.col("PRODUCT_NAME").like("%INVOKANA%") & (F.col("PRODUCT_STRENGTH") == "100 MG"), "INVOKANA 100MG")
           .when(F.col("PRODUCT_NAME").like("%INVOKANA%") & (F.col("PRODUCT_STRENGTH") == "300 MG"), "INVOKANA 300MG")
           .when(F.col("PRODUCT_NAME").like("%JARDIANCE%"), "JARDIANCE")
           .when(F.col("PRODUCT_NAME").like("%FARXIGA%"), "FARXIGA")
           .when(F.col("PRODUCT_NAME").isin(*DPP4_NAMES_30), "DPP-IV")
           .when(F.col("PRODUCT_NAME").isin(*GLP1_NAMES_30), "GLP-1")
           .otherwise(F.col("V_CAT")))
_prod = product_df.select("PRODUCT_NAME", "PRODUCT_STRENGTH").distinct()
_r30 = (_prod.join(ckd_mapping_df.select("PRODUCT_NAME", F.col("Treatment_Category").alias("V_CAT")), "PRODUCT_NAME")
             .filter(~F.lower("V_CAT").like("%other%"))
             .select("PRODUCT_NAME", "PRODUCT_STRENGTH", F.col("V_CAT").alias("CAT_30"), _lab30.alias("LABEL_30")))
_rb = (_prod.join(_b, "PRODUCT_NAME")
            .join(_o.withColumnRenamed("BRAND_LABEL", "O_LABEL"),
                  (_prod.PRODUCT_NAME == _o.PRODUCT_NAME) & (_prod.PRODUCT_STRENGTH == _o.STRENGTH), "left")
            .select(_prod.PRODUCT_NAME, _prod.PRODUCT_STRENGTH, F.col("TREATMENT_CATEGORY").alias("CAT_B"),
                    F.coalesce(F.expr("nullif(O_LABEL, '')"), F.expr("nullif(BRAND_LABEL, '')"), F.col("TREATMENT_CATEGORY")).alias("LABEL_B")))
_cmp = (_r30.join(_rb, ["PRODUCT_NAME", "PRODUCT_STRENGTH"], "full_outer")
        .withColumn("RESULT", F.when(F.col("CAT_30").isNull(), "only in the basket")
                               .when(F.col("CAT_B").isNull(), "only in 3.0")
                               .when((F.col("CAT_30") == F.col("CAT_B")) & (F.col("LABEL_30") == F.col("LABEL_B")), "same")
                               .otherwise("category or label differs")))
print("All products in the IQVIA product table (name + strength): 3.0 rule vs basket")
display(_cmp.groupBy("RESULT").agg(F.count("*").alias("NAME_STRENGTHS"), F.countDistinct("PRODUCT_NAME").alias("PRODUCT_NAMES")).orderBy("RESULT"))
_diff = _cmp.filter(F.col("RESULT") != "same")
_show(_diff.orderBy("PRODUCT_NAME").toPandas())

### 2e. Findings: basket vs 3.0

Run 2026-10-29. **The basket gives exactly the same result as the 3.0 view + typed lists**, for the validation HCPs and for every product in the IQVIA product table.

#### The new basket (`Final Product Basket/Kerendia_Rx_Final.csv`)

Built once by 2a (now switched off, `BUILD_RX_BASKET = False`), first as `Kerendia_Rx_Final_3_1_draft.csv`. Then renamed as for Lab, Dx and Px (D1): the original basket keeps its content under `Kerendia NBA 3.1 - RX Event List.csv`, and the new basket takes the name `Kerendia_Rx_Final.csv`.

| Rows | How they were made |
| --- | --- |
| 258 → 271 | Every row of the original basket. 13 clubbed rows were split in two, because their members follow different 3.0 rules (e.g. Lantus and Lantus Solostar are in the view as `Insulin`, the generic insulin glargine members are not) |
| + 94 | Products the 3.0 code keeps that were not in the basket (D2); `NOTE` = "Added (D2): in the view, not in the basket" |
| + 2 | Invokana brand label by strength (`100 MG` → `INVOKANA 100MG`, `300 MG` → `INVOKANA 300MG`) |
| **= 367** | 229 rows active in the journey (`ACTIVE_JOURNEY = Y`), 138 not (`N`, with the reason in `NOTE`: view category "Other" / "Other- glucose mgmt", or not in the view) |

* The 367 rows cover **440 product names** (346 from the original basket + 94 added). **254 are active**, the same 254 the 3.0 code keeps from the view.
* Journey categories copied from the view with the same texts, including `"metformin "` with its trailing space (checked by reading the file back).
* Brand labels: `JARDIANCE`, `FARXIGA`, `GLP-1` (Xultophy 100/3.6) on their rows, Invokana on the 2 strength rows. Every other active name uses its category as label, as in 3.0. DPP-IV and the other typed GLP-1 names are not active, so they carry no label (same as 3.0, see 1d).
* Some splits put both parts on `N` (e.g. FreeStyle Libre, Dexcom), because their members are inactive for different reasons ("Other" in the view vs not in the view). On split rows, `PATIENTS_TOTAL` is the club total, repeated on both parts.

#### Comparison with 3.0

| Check | 3.0 | Step 2 (basket) |
| --- | --- | --- |
| Fills (validation HCPs) | 126,903 | 126,903 |
| Patient-months | 107,616 | 107,616, all "same" (3,645 patients) |
| Patient-months per category and brand label | – | no change in any category |
| Patients with an SGLT2 month (PJ06 flag) | 902 | 902 (the same 902) |
| Every product in the IQVIA product table, by name + strength: kept or not, category, label | – | 685 name + strength pairs (221 product names) the same; none only in 3.0, none only in the basket, none different |

The all-products check covers every fill in the US, not only the validation HCPs: a fill reaches the journey only through its product name and strength. The 221 names are the 254 active names minus 33 that have no NDC in the product table (1d).

## Step 3: Run settings (`RX_CONFIG`) and one Snowflake query

**Changes:**
1. All table names, column names, dates, grace days and paths sit in one settings block, `RX_CONFIG` (3a), with checks that the tables and columns exist.
2. **One Snowflake query** with the filters (dates, basket products, NPI, the validation HCPs' patients) replaces the five full tables pulled into Spark (D7). The HCP universe is a Delta table, so it is still joined in Spark.
3. **Dates (D4)**, measured one at a time:
   * end date = `date_end` from the config, instead of the last date in the Rx table;
   * look-back: fills that start up to 97 days (90 days supply + 7 grace days) before `DATA_START_DATE` also count for the days on or after `DATA_START_DATE`.

| Cell | What | Compared with |
| --- | --- | --- |
| 3a | `RX_CONFIG` + checks | – |
| 3b | Query builder (basket + `RX_CONFIG`) and month logic (3.0 Steps 3, 5, 6, plus a cut at the start date for the look-back) | – |
| 3c | Run with the 3.0 dates (start `DATA_START_DATE`, end = last Rx claim date, no look-back) | Step 2: identical expected (only where the filters run changes) |
| 3d | Run with end = `date_end`, then with the look-back | the run before each |
| 3e | Findings | – |

In [0]:
# 3a. Run settings for the Rx journey. Change values here, not in the SQL.
RX_CONFIG = {
    "start_date": DATA_START_DATE,            # from 00_cohort_config (2023-01-01)
    "end_date": str(date_end)[:10],           # from 00_config (D4; 3.0 used the last date in the Rx table)
    "lookback_days": 97,                      # D4: fills starting up to 90 days supply + 7 grace days before start_date
    "grace_days": 7,                          # D5: days added after the days supply (3.0 value)
    "basket_path": RX_BASKET_PATH,            # Final Product Basket/Kerendia_Rx_Final.csv
    "hcp_universe_path": ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                          "hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"),
    "output_path": RX_VALIDATION_PATH,        # validation table (write switched off, Step 5)
    "tables": {
        "fact":     {"table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX",
                     "columns": {"patient_id": "PATIENT_ID", "service_date": "SVC_DT", "days_supply": "DAYS_SUPPLY",
                                 "ndc": "NDC_CD", "provider_id": "PROVIDER_ID", "plan_id": "PAYER_PLAN_ID"}},
        "product":  {"table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD",
                     "columns": {"ndc": "NDC_CD", "product_name": "PRODUCT_NAME", "strength": "PRODUCT_STRENGTH"}},
        "provider": {"table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR",
                     "columns": {"provider_id": "PROVIDER_ID", "npi": "NPI_NUMBER", "specialty": "PRIMARY_SPECIALTY_DESC"}},
        "plan":     {"table": "PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PLN",      # used until Step 4 (D6)
                     "columns": {"plan_id": "PAYER_PLAN_ID"}},
    },
}

# Checks: tables exist and every mapped column is there; dates in order; the HCP universe can be read
for _blk, _t in RX_CONFIG["tables"].items():
    _cols = set(c.upper() for c in Get_Data_Snowflakes(f"SELECT * FROM {_t['table']} LIMIT 0").columns)
    _miss = [c for c in _t["columns"].values() if c.upper() not in _cols]
    assert not _miss, f"{_t['table']}: columns not found {_miss}"
assert RX_CONFIG["start_date"] <= RX_CONFIG["end_date"], "start_date is after end_date"
_n_universe = spark.read.format("delta").load(RX_CONFIG["hcp_universe_path"]).select("PROVIDER_ID").distinct().count()
print(f"RX_CONFIG OK: {RX_CONFIG['start_date']} -> {RX_CONFIG['end_date']} (look-back {RX_CONFIG['lookback_days']} days, "
      f"grace {RX_CONFIG['grace_days']} days); tables and columns found; HCP universe {_n_universe:,} providers")

In [0]:
# 3b. The Rx query from the basket (2b) and RX_CONFIG (3a), and the month logic.

def _values(rows, n):
    # VALUES list for Snowflake; an empty list gives an empty table with n string columns
    if not rows:
        return "(SELECT " + ", ".join(f"NULL::STRING AS column{i + 1}" for i in range(n)) + " WHERE FALSE)"
    return "(VALUES " + ",\n".join("(" + ", ".join(_q(v) for v in r) + ")" for r in rows) + ")"

def build_rx_query(cfg, base, overrides, end_date, lookback_days=0, plan_join=True, tag_drops=False, npis=None):
    """One Snowflake query: fills of the active basket products, with journey category and brand label.
    tag_drops=False: rows that fail a rule are filtered out (Step 3).
    tag_drops=True: they are kept with a DROP_REASON (Step 4 dropped-rows report).
    npis: validation HCPs -> only the patients with a fill from them since start_date."""
    T = cfg["tables"]
    fc, pc, vc, lc = (T[k]["columns"] for k in ["fact", "product", "provider", "plan"])
    start, grace = cfg["start_date"], cfg["grace_days"]
    pull_start = (date.fromisoformat(start) - timedelta(days=lookback_days)).isoformat()
    ds = f"ROUND(TRY_TO_DECIMAL(TO_VARCHAR(F.{fc['days_supply']}), 38, 6))"   # days supply as a number (stored as text)
    plan_sql = f"JOIN {T['plan']['table']} L ON F.{fc['plan_id']} = L.{lc['plan_id']}" if plan_join else ""
    prov_join = "LEFT JOIN" if tag_drops else "JOIN"
    reason = f"""CASE WHEN V.{vc['provider_id']} IS NULL THEN '1. provider not in the provider table'
                WHEN V.{vc['npi']} IS NULL THEN '2. provider has no NPI'
                WHEN F.{fc['service_date']} > '{end_date}' THEN '3. fill after the end date'
                WHEN {ds} IS NULL THEN '4. days supply empty'
                WHEN F.{fc['service_date']} < '{start}' AND DATEADD(day, {ds} + {grace}, F.{fc['service_date']}) < '{start}'
                     THEN '5. fill before the start date that does not reach it'
                ELSE NULL END"""
    where = [f"F.{fc['service_date']} >= '{pull_start}'"]
    if not tag_drops:
        where += [f"F.{fc['service_date']} <= '{end_date}'", f"V.{vc['npi']} IS NOT NULL"]
    if npis:
        where.append(f"""F.{fc['patient_id']} IN (
        SELECT F2.{fc['patient_id']} FROM {T['fact']['table']} F2
        JOIN {T['product']['table']} P2 ON F2.{fc['ndc']} = P2.{pc['ndc']}
        JOIN B ON P2.{pc['product_name']} = B.PRODUCT_NAME
        JOIN {T['provider']['table']} V2 ON F2.{fc['provider_id']} = V2.{vc['provider_id']}
        WHERE V2.{vc['npi']} IN ({', '.join(_q(n) for n in npis)}) AND F2.{fc['service_date']} >= '{start}')""")
    return f"""
WITH B AS (SELECT column1 AS PRODUCT_NAME, column2 AS TREATMENT_CATEGORY, column3 AS BRAND_LABEL
           FROM {_values(base[['PRODUCT_NAME', 'TREATMENT_CATEGORY', 'BRAND_LABEL']].values.tolist(), 3)}),
     O AS (SELECT column1 AS PRODUCT_NAME, column2 AS STRENGTH, column3 AS BRAND_LABEL
           FROM {_values(overrides[['PRODUCT_NAME', 'STRENGTH', 'BRAND_LABEL']].values.tolist(), 3)})
SELECT F.{fc['patient_id']} AS PATIENT_ID, F.{fc['service_date']} AS SVC_DT, F.{fc['days_supply']} AS DAYS_SUPPLY,
       F.{fc['provider_id']} AS PROVIDER_ID, P.{pc['product_name']} AS PRODUCT_NAME, P.{pc['strength']} AS PRODUCT_STRENGTH,
       V.{vc['npi']} AS NPI_NUMBER, V.{vc['specialty']} AS PRIMARY_SPECIALTY_DESC,
       B.TREATMENT_CATEGORY,
       COALESCE(NULLIF(O.BRAND_LABEL, ''), NULLIF(B.BRAND_LABEL, ''), B.TREATMENT_CATEGORY) AS CATEGORY
       {(', ' + reason + ' AS DROP_REASON') if tag_drops else ''}
FROM {T['fact']['table']} F
JOIN {T['product']['table']} P ON F.{fc['ndc']} = P.{pc['ndc']}
JOIN B ON P.{pc['product_name']} = B.PRODUCT_NAME
LEFT JOIN O ON P.{pc['product_name']} = O.PRODUCT_NAME AND P.{pc['strength']} = O.STRENGTH
{prov_join} {T['provider']['table']} V ON F.{fc['provider_id']} = V.{vc['provider_id']}
{plan_sql}
WHERE {' AND '.join(where)}
"""

def rx_validation_patients(df, cfg, npis):
    # 3.0 apply_hcp_filter: patients with a kept fill (since start_date) from a validation HCP; all their kept fills
    if not npis:
        return df
    pats = (df.filter((F.col("SVC_DT") >= F.lit(cfg["start_date"])) & F.col("NPI_NUMBER").isin(npis))
              .select("PATIENT_ID").distinct())
    return df.join(pats, "PATIENT_ID", "left_semi")

def rx_month_journey_v31(rx_fills, start_date, end_date, grace_days):
    # 3.0 Steps 3, 5, 6 (as rx_month_journey in 2c) + days before start_date cut (only matters with the look-back)
    e = (rx_fills.withColumn("DAYS_SUPPLY_T", F.round(F.col("DAYS_SUPPLY")).cast("int"))
                 .withColumn("END_DT", F.least(F.expr(f"date_add(SVC_DT, DAYS_SUPPLY_T + {grace_days})"), F.lit(end_date).cast("date"))))
    x = (e.withColumn("TXN_DATE", F.explode(F.expr("sequence(SVC_DT, END_DT, interval 1 day)")))
          .filter(F.col("TXN_DATE") >= F.lit(start_date).cast("date"))
          .withColumn("COHORT_MONTH", F.date_format(assign_cohort_month(F.col("TXN_DATE")), "yyyy-MM-dd")))
    m = (x.groupBy("PATIENT_ID", "COHORT_MONTH")
          .agg(F.collect_set("TREATMENT_CATEGORY").alias("TREATMENT_CATEGORY"), F.collect_set("CATEGORY").alias("CATEGORY")))
    m = m.join(df_cohort_ref.select(F.date_format("cohort_ref_month", "yyyy-MM-dd").alias("_ref_month_str"), "cohort_label"),
               m.COHORT_MONTH == F.col("_ref_month_str"), "left").drop("_ref_month_str")
    return m.withColumnRenamed("COHORT_MONTH", "DATE").withColumn("DATE", F.to_date("DATE", "yyyy-MM-dd"))

def run_rx_step3(cfg, end_date, lookback_days):
    # Snowflake query -> HCP universe (Spark) -> validation patients -> months. Returns (fills, months, rows pulled)
    npis = get_test_npis()
    pulled = Get_Data_Snowflakes(build_rx_query(cfg, rx_basket_base, rx_basket_overrides, end_date, lookback_days, plan_join=True, npis=npis))
    hcp = spark.read.format("delta").load(cfg["hcp_universe_path"]).select("PROVIDER_ID").distinct()
    fills = rx_validation_patients(pulled.join(hcp, "PROVIDER_ID", "left_semi"), cfg, npis)
    return fills, rx_month_journey_v31(fills, cfg["start_date"], end_date, cfg["grace_days"]), pulled.count()

print(build_rx_query(RX_CONFIG, rx_basket_base.head(3), rx_basket_overrides, RX_CONFIG["end_date"], npis=get_test_npis()[:2]))
print("(query shown with 3 basket products and 2 HCPs only)")

In [0]:
# 3c. Same dates as 3.0 (end = last Rx claim date, no look-back): only where the filters run changes.
rx_fills_s3a, df_rx_s3a, _pulled_s3a = run_rx_step3(RX_CONFIG, str(max_date), 0)
print(f"Rows pulled from Snowflake: {_pulled_s3a:,} (3.0 pulled the full Rx fact table: {fact_rx_df.count():,} rows, plus 4 full tables)")
print(f"Fills: Step 3 {rx_fills_s3a.count():,} vs Step 2 {rx_fills_s2.count():,}; patient-months: {df_rx_s3a.count():,} vs {df_rx_s2.count():,}")
_s3a, _c3a, _g3a = compare_rx(df_rx_s2, df_rx_s3a)
print("Per patient-month (old = Step 2, new = Step 3 with the 3.0 dates)")
display(_s3a)
display(_c3a[_c3a.CHANGE != 0] if (_c3a.CHANGE != 0).any() else pd.DataFrame([{"PER_CATEGORY": "no change in any category"}]))
display(_g3a)

In [0]:
# 3d. D4, one change at a time: (1) end date = date_end, (2) + look-back of 97 days.
rx_fills_s3b, df_rx_s3b, _ = run_rx_step3(RX_CONFIG, RX_CONFIG["end_date"], 0)
rx_fills_s3, df_rx_s3, _ = run_rx_step3(RX_CONFIG, RX_CONFIG["end_date"], RX_CONFIG["lookback_days"])
print(f"End date: 3.0 {max_date} -> date_end {RX_CONFIG['end_date']}; look-back {RX_CONFIG['lookback_days']} days")
print(f"Fills: 3.0 dates {rx_fills_s3a.count():,} | end = date_end {rx_fills_s3b.count():,} | + look-back {rx_fills_s3.count():,}")
print(f"Patient-months: {df_rx_s3a.count():,} | {df_rx_s3b.count():,} | {df_rx_s3.count():,}")

for _name, _old, _new in [("(1) end date = date_end (old = 3.0 dates)", df_rx_s3a, df_rx_s3b),
                          ("(2) + look-back (old = end date = date_end)", df_rx_s3b, df_rx_s3)]:
    _s, _c, _g = compare_rx(_old, _new)
    print(_name)
    display(_s)
    display(_c[_c.CHANGE != 0] if (_c.CHANGE != 0).any() else pd.DataFrame([{"PER_CATEGORY": "no change in any category"}]))
    display(_g)

# Which months change: patient-months per DATE, 3.0 dates vs Step 3 final
display(df_rx_s3a.groupBy("DATE").count().withColumnRenamed("count", "MONTHS_30_DATES")
        .join(df_rx_s3.groupBy("DATE").count().withColumnRenamed("count", "MONTHS_STEP3"), "DATE", "full_outer")
        .fillna(0).withColumn("CHANGE", F.col("MONTHS_STEP3") - F.col("MONTHS_30_DATES"))
        .filter(F.col("CHANGE") != 0).orderBy("DATE"))

### 3e. Findings: config and dates

Run 2026-10-29, validation HCPs.

**1. Settings and one Snowflake query (3a–3c): identical to Step 2.**

* `RX_CONFIG` checks passed: all 4 tables and their mapped columns found; 2023-01-01 → 2026-09-30; HCP universe 234,405 providers.
* **Rows pulled: 160,029** (the fills of the validation patients). The 3.0 code pulled the full Rx fact table (**5,989,489,699 rows**) plus 4 full tables into Spark.
* With the 3.0 dates (end = last Rx claim date 2026-10-02, no look-back): 126,903 fills and 107,616 patient-months, all "same" as Step 2. No change in any category, the same 902 SGLT2 patients.

**2. End date = `date_end` (2026-09-30 instead of 2026-10-02), D4:**

| | 3.0 dates | End = `date_end` |
| --- | --- | --- |
| Fills | 126,903 | 126,561 (−342: fills on 1–2 Oct 2026) |
| Patient-months | 107,616 | 107,524 |
| Result | – | 107,508 same, 16 with fewer categories, 92 gone (5 patients) |
| SGLT2 patients | 902 | 900 |

The 92 months that disappear are spread over all months (−2 in each month from 2023-04 on). That points to 2 patients who leave the validation set: their only validation-HCP fill was on 1–2 Oct 2026, after the new end date. The last month (2026-09) loses 5 patient-months.

**3. + look-back of 97 days, D4:**

| | End = `date_end` | + look-back |
| --- | --- | --- |
| Fills | 126,561 | 134,389 (+7,828 from Sep–Dec 2022) |
| Patient-months | 107,524 | 110,049 (+2,525 new months; 1,247 months gain a category; 2 change a label) |
| SGLT2 patients | 900 | 909 (+9) |

Only the first months change, as expected:

| Cohort month | Before | After | Change |
| --- | --- | --- | --- |
| 2022-12 (days from 1 Jan 2023 that fall in this cohort month) | 375 | 1,976 | +1,601 |
| 2023-01 | 1,466 | 2,103 | +637 |
| 2023-02 | 1,901 | 2,139 | +238 |
| 2023-03 | 2,160 | 2,204 | +44 |
| 2023-04 onwards | – | – | 0 |

Without the look-back, the first months of 2023 counted only fills made from 1 Jan 2023, so a patient on a 90-day fill from December 2022 looked untreated until the next refill. Days before 2023-01-01 are still cut (the journey still starts on `start_date`). By category the gain is largest for Calcium channel / Beta blocker (+1,721 months), ACE/ ARB (+1,205), Diuretic_NON_MRA (+961), Combo drug (+822) and metformin (+676); SGLT2 +237.

## Step 4: Code clean-up and dropped-rows report

**Changes (D6, R11):**
1. The plan join is removed (Step 1: it removes 0 fills and no plan column is used). The demographics join is gone since Step 3 (it was a left join with no column used).
2. Fills with an **empty days supply are dropped** instead of being spread to the end of the data (Step 1, 1d point 7).
3. Every fill of the validation patients that the rules remove is kept with a reason and counted (**dropped-rows report**), instead of being filtered silently.

Output columns unchanged. Compared with Step 3: identical expected for the validation HCPs (0 empty days supply there, plan join removes 0).

In [0]:
# 4a. Same query as Step 3 (end = date_end, look-back), without the plan join, every row tagged with DROP_REASON.
_npis = get_test_npis()
_pulled4 = Get_Data_Snowflakes(build_rx_query(RX_CONFIG, rx_basket_base, rx_basket_overrides, RX_CONFIG["end_date"],
                                              RX_CONFIG["lookback_days"], plan_join=False, tag_drops=True, npis=_npis))
_hcp = (spark.read.format("delta").load(RX_CONFIG["hcp_universe_path"]).select("PROVIDER_ID").distinct()
        .withColumn("_IN_UNIVERSE", F.lit(1)))
rx_fills_tagged = (_pulled4.join(_hcp, "PROVIDER_ID", "left")
                   .withColumn("DROP_REASON", F.coalesce("DROP_REASON",
                               F.when(F.col("_IN_UNIVERSE").isNull(), F.lit("6. provider not in the HCP universe"))))
                   .drop("_IN_UNIVERSE"))
# Validation patients from the kept fills (as 3.0), then all their fills, kept and dropped
_pats4 = (rx_fills_tagged.filter(F.col("DROP_REASON").isNull() & (F.col("SVC_DT") >= F.lit(RX_CONFIG["start_date"]))
                                 & F.col("NPI_NUMBER").isin(_npis)).select("PATIENT_ID").distinct()) if _npis else None
rx_fills_tagged = rx_fills_tagged.join(_pats4, "PATIENT_ID", "left_semi") if _npis else rx_fills_tagged
rx_fills_s4 = rx_fills_tagged.filter(F.col("DROP_REASON").isNull()).drop("DROP_REASON")
df_rx_s4 = rx_month_journey_v31(rx_fills_s4, RX_CONFIG["start_date"], RX_CONFIG["end_date"], RX_CONFIG["grace_days"])
print(f"Fills of the validation patients: {rx_fills_tagged.count():,}; kept {rx_fills_s4.count():,}; patient-months {df_rx_s4.count():,}")

In [0]:
# 4b. (1) vs Step 3; (2) dropped-rows report (validation patients), by reason and by journey category.
_s4, _c4, _g4 = compare_rx(df_rx_s3, df_rx_s4)
print("Per patient-month (old = Step 3, new = Step 4)")
display(_s4)
display(_c4[_c4.CHANGE != 0] if (_c4.CHANGE != 0).any() else pd.DataFrame([{"PER_CATEGORY": "no change in any category"}]))
display(_g4)
print(f"Fills: Step 4 {rx_fills_s4.count():,} vs Step 3 {rx_fills_s3.count():,}")

print("Dropped-rows report: fills of the validation patients removed by each rule (first rule that applies)")
rx_drop_report = (rx_fills_tagged.withColumn("REASON", F.coalesce("DROP_REASON", F.lit("0. kept")))
                  .groupBy("REASON").agg(F.count("*").alias("FILLS"), F.countDistinct("PATIENT_ID").alias("PATIENTS"))
                  .orderBy("REASON").toPandas())
rx_drop_report["SHARE_PCT"] = (100 * rx_drop_report.FILLS / rx_drop_report.FILLS.sum()).round(2)
display(rx_drop_report)
display(rx_fills_tagged.filter(F.col("DROP_REASON").isNotNull()).groupBy("DROP_REASON", "TREATMENT_CATEGORY").count()
        .orderBy("DROP_REASON", F.desc("count")))

In [0]:
# 4b-2. Second check of the fill counts: which fills are only in Step 3 or only in Step 4, and why.
_k = ["PATIENT_ID", "PROVIDER_ID", "SVC_DT", "DAYS_SUPPLY", "PRODUCT_NAME", "PRODUCT_STRENGTH"]
_only3 = rx_fills_s3.select(*_k).exceptAll(rx_fills_s4.select(*_k))
_only4 = rx_fills_s4.select(*_k).exceptAll(rx_fills_s3.select(*_k))
_start = F.lit(RX_CONFIG["start_date"]).cast("date")
print(f"Fills only in Step 3: {_only3.count():,}; only in Step 4: {_only4.count():,}")
for _n, _d in [("only in Step 3", _only3), ("only in Step 4", _only4)]:
    print(_n)
    _show(_d.withColumn("WHEN", F.when(F.col("SVC_DT") < _start, "before start_date (look-back)").otherwise("from start_date"))
            .groupBy("WHEN").count().toPandas())

# Fills only in Step 4 from the look-back: do they have a plan in the plan table? (Step 3 still had the plan join)
_ids = [r.PATIENT_ID for r in _only4.select("PATIENT_ID").distinct().collect()]
if _ids:
    T = RX_CONFIG["tables"]
    display(Get_Data_Snowflakes(f"""
    SELECT IFF(L.PAYER_PLAN_ID IS NULL, 'plan not in the plan table', 'plan found') AS PLAN, COUNT(*) AS FILLS
    FROM {T['fact']['table']} F
    JOIN {T['product']['table']} P ON F.NDC_CD = P.NDC_CD
    JOIN (SELECT column1 AS PRODUCT_NAME FROM {_values([[n] for n in rx_basket_base.PRODUCT_NAME], 1)}) B ON P.PRODUCT_NAME = B.PRODUCT_NAME
    LEFT JOIN {T['plan']['table']} L ON F.PAYER_PLAN_ID = L.PAYER_PLAN_ID
    WHERE F.PATIENT_ID IN ({', '.join(_q(i) for i in _ids)})
      AND F.SVC_DT >= DATEADD(day, -{RX_CONFIG['lookback_days']}, '{RX_CONFIG['start_date']}') AND F.SVC_DT < '{RX_CONFIG['start_date']}'
    GROUP BY 1"""))

### 4c. Findings: clean-up and dropped rows

Run 2026-10-29, validation HCPs.

**Output identical to Step 3:** 110,049 patient-months, all "same" (3,643 patients), no change in any category, the same 909 SGLT2 patients. Removing the plan join and the demographics join changes nothing (as measured in Step 1), and no fill has an empty days supply here.

**Dropped-rows report** (every fill of the validation patients pulled from Snowflake, by the first rule that removes it; the rows add up to the 158,983 fills pulled):

| Rule | Fills | Share | Patients |
| --- | --- | --- | --- |
| Kept | 131,949 | 83.0% | 3,643 |
| Provider not in the provider table | 0 | 0% | 0 |
| Provider has no NPI | 216 | 0.14% | 91 |
| Fill after the end date (2026-09-30) | 213 | 0.13% | 149 |
| Days supply empty (dropped, D4 / 1d) | 0 | 0% | 0 |
| Fill before the start date that does not reach it (look-back) | 3,094 | 1.95% | 956 |
| Prescriber not in the HCP universe | 23,511 | 14.79% | 2,004 |

* The biggest rule is the **HCP universe** (3.0 rule, unchanged): 14.8% of the validation patients' fills were written by a prescriber outside the universe. It is now visible instead of silent.
* **Second check of the counts (4b-2):** every Step 4 fill is also in Step 3 (0 only in Step 4). The 2,440 fills only in Step 3 are all look-back fills that end before 2023, so Step 3 pulled them but they added no day; Step 4 drops them up front. The other 654 of the 3,094 "before the start date" fills are fills Step 3 also removed through another rule (they are counted under the first rule that applies).
* Empty days supply: 0 here; for all US 2,686 kept fills (1c). They are now dropped and reported instead of being spread to the end of the data.

## Step 5: Write the output (switched off)

The cell writes the Step 4 output (`df_rx_final`, same 5 columns as 3.0, checked). `WRITE_OUTPUT = False` until the Dx and Px 3.1 tables go live (plan D8). `06_patient_journey_combine` keeps reading the table written by the 3.0 notebook on 2026-10-26.

In [0]:
# Step 5. Write the Rx journey (VALIDATION path, not production).
# End-to-end run 2026-10-30: all 3.1 journeys write to their own dated folder (RUN_OUTPUT_DIR), so the
# *_validation tables that 06_patient_journey_combine and the HCP notebooks read today are not touched (plan D8).
# Output = Step 4 (basket + RX_CONFIG + clean-up); same 5 columns as 3.0.
WRITE_OUTPUT = True
RUN_OUTPUT_DIR = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/"
                  "hcp_level_model/ops_pipeline/query_output/validation_2026_10_30/")
RX_CONFIG["output_path"] = RUN_OUTPUT_DIR + "kerendia_nba_3_1_rx_journey_validation"
df_rx_final = df_rx_s4.select("PATIENT_ID", "DATE", "TREATMENT_CATEGORY", "CATEGORY", "cohort_label")
assert df_rx_final.columns == df_rx_30.columns, f"Output columns differ from 3.0: {df_rx_final.columns}"

if WRITE_OUTPUT:
    df_rx_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(RX_CONFIG["output_path"])
    _w = spark.read.format("delta").load(RX_CONFIG["output_path"])
    print(f"Written: {RX_CONFIG['output_path']} | {_w.count():,} patient-months, {_w.select('PATIENT_ID').distinct().count():,} patients")
else:
    print("WRITE_OUTPUT = False: nothing written. The table still holds the 3.0 output of 2026-10-26.")